# 5. Data-quality diagnostics

Target encoding, rates by year, linked-report check with a 70 / 15 / 15 split preview, and repeated synopses.

### Run the earlier steps
Loads the variables this notebook needs (`df`, `df_norm`, ...) by running `04_candidate_features.ipynb` (which runs the steps before it). Output is hidden.

In [1]:
%%capture
%run -i "04_candidate_features.ipynb"

## 5.1 Diagnostic 1: target encoding and imbalance

In [2]:
# Diagnostic 1: target encoding and overall imbalance

if not isinstance(globals().get("df"), pd.DataFrame) or df.empty:
    raise ValueError(
        "Run the CSV-loading cell first so that df contains the reports."
    )

# Work on a small copy. Do not change df, df_norm, or the original CSV.
headers = pd.Index([" ".join(str(c).split()) for c in df.columns])

if headers.has_duplicates:
    raise ValueError(
        "Duplicate column names found after whitespace normalization."
    )

TARGET = "Events Anomaly / Safety Concern"
required = ["ACN", "Time Date", TARGET, "Synopsis"]

missing = [c for c in required if c not in headers]
if missing:
    raise KeyError(f"Required columns not found: {missing}")

keep = (
    headers.isin(required)
    | headers.str.startswith("Person C3RS Report Number /")
)
diag = df.loc[:, keep].copy().reset_index(drop=True)
diag.columns = headers[keep]

print(f"Source dataframe: {len(df):,} reports, {df.shape[1]:,} columns")
print("\nTarget values before diagnostic encoding:")
print(diag[TARGET].value_counts(dropna=False).to_string())

value = diag[TARGET].astype("string").str.strip().str.lower()

status = pd.Series("Other / unresolved", index=diag.index)
status.loc[value.isna() | value.eq("").fillna(False)] = "Blank"
status.loc[value.isin(["true", "1", "1.0", "yes"])] = "Selected"
status.loc[value.isin(["false", "0", "0.0", "no"])] = "Explicit false / zero"

label_counts = (
    status.value_counts()
    .reindex(
        ["Selected", "Explicit false / zero", "Blank", "Other / unresolved"],
        fill_value=0,
    )
    .rename_axis("Export encoding")
    .to_frame("Records")
)

label_counts["Percent of all reports"] = (
    100 * label_counts["Records"] / len(diag)
)
display(label_counts.round(2))

if status.eq("Other / unresolved").any():
    raise ValueError(
        "Unrecognized target values remain. Share the output before continuing."
    )

# ONLY for the proposal's selected-versus-not-selected CODE classification.
# A blank is not evidence that the situation was safe.
diag_y = status.eq("Selected").astype("int64")

print(f"\nSelected-code percentage: {100 * diag_y.mean():.2f}%")

if diag_y.nunique() == 2:
    print(
        "All-not-selected baseline accuracy: "
        f"{100 * (1 - diag_y.mean()):.2f}%"
    )
    print("That baseline identifies none of the selected reports (recall = 0%).")

Source dataframe: 10,261 reports, 2,040 columns

Target values before diagnostic encoding:
Events Anomaly / Safety Concern
NaN     9850
True     411


,Records,Percent of all reports
Export encoding,,
Selected,411,4.01
Explicit false / zero,0,0.00
Blank,9850,95.99
Other / unresolved,0,0.00



Selected-code percentage: 4.01%
All-not-selected baseline accuracy: 95.99%
That baseline identifies none of the selected reports (recall = 0%).


## 5.2 Diagnostic 2: target distribution by year

In [3]:
# Diagnostic 2: target distribution by year
month = pd.to_datetime(
    diag["Time Date"].astype("string").str.strip(),
    format="%Y-%m",
    errors="coerce",
)

by_year = (
    pd.DataFrame({
        "Year": month.dt.year.astype("Int64"),
        "Month": month.dt.month.astype("Int64"),
        "Selected": diag_y,
    })
    .groupby("Year", dropna=False)
    .agg(
        Reports=("Selected", "size"),
        Selected=("Selected", "sum"),
        Months_present=("Month", "nunique"),
    )
)

by_year["Not_selected"] = by_year["Reports"] - by_year["Selected"]
by_year["Selected_pct"] = (
    100 * by_year["Selected"] / by_year["Reports"]
)

display(
    by_year[
        ["Reports", "Selected", "Not_selected", "Selected_pct", "Months_present"]
    ].round(2)
)

print(f"\nMissing or unparsed dates: {month.isna().sum():,}")

if month.isna().any():
    display(diag.loc[month.isna(), ["ACN", "Time Date"]].head(10))

print("These are code-selection percentages within the export, not accident rates.")

,Reports,Selected,Not_selected,Selected_pct,Months_present
Year,,,,,
2016,411,30,381,7.30,12
2017,344,24,320,6.98,12
2018,578,32,546,5.54,12
2019,746,31,715,4.16,12
2020,649,29,620,4.47,12
2021,1141,23,1118,2.02,12
2022,1128,26,1102,2.30,12
2023,1300,60,1240,4.62,12
2024,1338,41,1297,3.06,12



Missing or unparsed dates: 0
These are code-selection percentages within the export, not accident rates.


## 5.3 Diagnostic 3: linked reports and 70 / 15 / 15 split preview

In [4]:
# Diagnostic 3: proposed 70/15/15 split -- preview only, no model training

# Check whether an accession number connects more than one row.
# Repeated references within the SAME row do not count as separate records.
primary_ids = diag["ACN"].astype("string").str.strip()

if primary_ids.fillna("").eq("").any():
    raise ValueError(
        "Missing primary ACNs found. Review identifiers before splitting."
    )

id_columns = ["ACN"] + [
    c for c in diag
    if c.startswith("Person C3RS Report Number /")
]

id_long = diag[id_columns].assign(Row=diag.index).melt(
    id_vars="Row",
    value_name="Accession",
)

id_long["Accession"] = (
    id_long["Accession"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

id_long = id_long.loc[id_long["Accession"].fillna("").ne("")]

shared_ids = id_long.groupby("Accession")["Row"].nunique()
shared_ids = shared_ids[shared_ids > 1]

print(f"Accession numbers appearing in multiple rows: {len(shared_ids):,}")

if len(shared_ids):
    display(shared_ids.head(10).rename("Rows"))
    raise ValueError(
        "Linked reports found. We need a grouped split before proceeding."
    )

if diag_y.nunique() != 2:
    raise ValueError(
        "The proposed split needs both selected and not-selected records."
    )

# Separate variable names keep this preview apart from the modeling pipeline.
diag_train, diag_holdout = train_test_split(
    diag.index.to_numpy(),
    test_size=0.30,
    stratify=diag_y.to_numpy(),
    random_state=42,
)

diag_val, diag_test = train_test_split(
    diag_holdout,
    test_size=0.50,
    stratify=diag_y.iloc[diag_holdout].to_numpy(),
    random_state=42,
)

diag_partitions = {
    "Training": diag_train,
    "Validation": diag_val,
    "Test": diag_test,
}

# Verify that each row appears exactly once.
all_positions = np.concatenate(list(diag_partitions.values()))
assert len(all_positions) == len(diag)
assert len(np.unique(all_positions)) == len(diag)

# Compare summaries without changing the original text.
summary_key = (
    diag["Synopsis"]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

training_summaries = set(summary_key.iloc[diag_train]) - {""}

rows = []

for split_name, positions in diag_partitions.items():
    labels = diag_y.iloc[positions]
    summaries = summary_key.iloc[positions]
    selected = int(labels.sum())

    rows.append({
        "Split": split_name,
        "Reports": len(positions),
        "Selected": selected,
        "Not_selected": len(positions) - selected,
        "Selected_pct": 100 * labels.mean(),
        "Blank_synopsis": int(summaries.eq("").sum()),
        "Synopsis_seen_in_training": (
            int(summaries.isin(training_summaries).sum())
            if split_name != "Training"
            else pd.NA
        ),
    })

split_counts = pd.DataFrame(rows).set_index("Split")
display(split_counts.round(2))

print("\nRepeated synopses are a review flag, not proof of duplicate events.")
print("This preview does not establish a leakage-free final split.")
print("No rows were removed and no model was fitted.")

Accession numbers appearing in multiple rows: 0


,Reports,Selected,Not_selected,Selected_pct,Blank_synopsis,Synopsis_seen_in_training
Split,,,,,,
Training,7182,288,6894,4.01,0,<NA>
Validation,1539,61,1478,3.96,0,407
Test,1540,62,1478,4.03,0,435



Repeated synopses are a review flag, not proof of duplicate events.
This preview does not establish a leakage-free final split.
No rows were removed and no model was fitted.


## 5.4 Diagnostic 4: same synopsis, different codes

In [5]:
# Diagnostic 4: same Synopsis, different Safety Concern codes
# Inspect TRAINING reports only. Do not change df or the split.

required = ["df", "diag", "diag_y", "diag_partitions"]
missing = [name for name in required if name not in globals()]
if missing:
    raise ValueError(f"Run Diagnostics 1-3 first. Missing: {missing}")

column_map = {" ".join(str(c).split()): c for c in df.columns}
narrative_name = "Person Narrative / Narrative 1"

if narrative_name not in column_map or "ACN" not in column_map:
    raise KeyError("The first narrative or ACN column was not found.")

# Verify that row positions still match the earlier diagnostics.
source_acn = (
    df[column_map["ACN"]].reset_index(drop=True).astype("string")
)
if not source_acn.equals(diag["ACN"].astype("string")):
    raise ValueError("df changed after Diagnostics 1-3. Rerun those cells.")

positions = diag_partitions["Training"]

text_review = diag.loc[
    positions, ["ACN", "Time Date", "Synopsis"]
].copy()

text_review["Selected"] = diag_y.loc[positions]
text_review["Narrative"] = (
    df[column_map[narrative_name]]
    .reset_index(drop=True)
    .loc[positions]
)

# Normalize only for matching; preserve the original report text.
text_review["Summary_key"] = (
    text_review["Synopsis"].astype("string").fillna("")
    .str.strip().str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

groups = (
    text_review.loc[text_review["Summary_key"].ne("")]
    .groupby("Summary_key")
    .agg(
        Records=("Selected", "size"),
        Selected=("Selected", "sum"),
        Synopsis=("Synopsis", "first"),
    )
)
groups["Not_selected"] = groups["Records"] - groups["Selected"]

mixed = groups.loc[
    groups["Selected"].gt(0) & groups["Not_selected"].gt(0)
].sort_values("Records", ascending=False)

print(f"Training synopses appearing with both codes: {len(mixed):,}")

with pd.option_context("display.max_colwidth", None):
    display(
        mixed[["Records", "Selected", "Not_selected", "Synopsis"]]
        .head(3)
        .reset_index(drop=True)
    )

for number, key in enumerate(mixed.head(3).index, start=1):
    group = text_review.loc[text_review["Summary_key"].eq(key)]

    print(f"\n{'=' * 65}\nExample group {number}")
    print("Shared Synopsis:", group["Synopsis"].iloc[0])

    for label in [1, 0]:
        example = group.loc[group["Selected"].eq(label)].iloc[0]
        text = (
            ""
            if pd.isna(example["Narrative"])
            else str(example["Narrative"])
        )

        print(
            f"\nACN {example['ACN']} | {example['Time Date']} "
            f"| Code {label}"
        )
        print(text[:1200])

        if len(text) > 1200:
            print("[Excerpt truncated after 1,200 characters.]")

print(
    "\nDifferent codes for the same summary do not, by themselves, "
    "prove a coding error."
)

Training synopses appearing with both codes: 7


,Records,Selected,Not_selected,Synopsis
0,99,1,98,"An Engineer reported exceeding Maximum Authorized Speed, resulting in a Positive Train Control penalty brake application."
1,44,1,43,"An Engineer exceeded Maximum Authorized Speed, resulting in a Positive Train Control penalty brake application."
2,33,1,32,An Engineer reported operating overspeed in a Temporary Speed Restriction.



Example group 1
Shared Synopsis: An Engineer reported exceeding Maximum Authorized Speed, resulting in a Positive Train Control penalty brake application.

ACN 38077 | 2025-07 | Code 1
Assigned to Job X. [We] had to bring ten fully loaded ballast cars with one engine on each end from Yard X to Subdivision X. First, [we] had to make the train up in Yard X. At first, [we] used Engine A (Type A) to bring ten ballast cars from Track X to Track Y. This was a challenge to use just one engine for a few reasons. The limited number of working Type A engines. The increased grade as [the] train proceeds towards Station X end of Yard X to clear the switch for Track Y. The train began to slow down. The Maximum Authorized Speed (MAS) in Yard X is Restricted Speed not exceeding X MPH, which at times, you have to go faster to use the momentum of the train to get you moving up the increased grade. The limitations on Engine A, which was built in the 1970s, even going 3 MPH to 5 MPH above the MAS (X MPH